In [2]:
# Next setting up this new notebook now for running the same model once again but now this time on AWS Sagemaker
# This should follow similar methodology to the part b pattern we'd used in assignment 5
import os
import boto3
import sagemaker
from sagemaker.estimator import Estimator
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer

session = sagemaker.Session()
region = boto3.Session().region_name
role = sagemaker.get_execution_role()
bucket = session.default_bucket()
prefix = 'ana680-bgg-sagemaker-byoc'

print('Region:', region)
print('Role:', role)
print('Bucket:', bucket)

Region: us-east-2
Role: arn:aws:iam::047557961235:role/service-role/AmazonSageMaker-ExecutionRole-20260221T114423
Bucket: sagemaker-us-east-2-047557961235


In [4]:
!pwd

/home/ec2-user/SageMaker/Board-Game-Ratings-Predictions/sagemaker/notebooks


In [5]:
# Paths are inside this sagemaker folder so this setup is self-contained
local_train_csv = '../data/bgg_master.csv'
if not os.path.exists(local_train_csv):
    raise FileNotFoundError(f'Missing training data at {local_train_csv}')

train_s3_uri = session.upload_data(path=local_train_csv, bucket=bucket, key_prefix=f'{prefix}/train')
print('Train S3 URI:', train_s3_uri)


Train S3 URI: s3://sagemaker-us-east-2-047557961235/ana680-bgg-sagemaker-byoc/train/bgg_master.csv


In [6]:
# Ensure ECR repo exists or create one if it doesn't yet (Amazon Elastic Container Registry repository)
# This will be how we incorporate the docker image, gotta build it here, then push the image to the ECR repo, 
# And SageMaker pulls it for training and predictions
repo_name = 'ana680-bgg-ratings'
ecr = boto3.client('ecr', region_name=region)
try:
    ecr.create_repository(repositoryName=repo_name)
    print('Created ECR repo:', repo_name)
except ecr.exceptions.RepositoryAlreadyExistsException:
    print('ECR repo already exists:', repo_name)

account_id = boto3.client('sts').get_caller_identity()['Account']
image_uri = f'{account_id}.dkr.ecr.{region}.amazonaws.com/{repo_name}:latest'
print('Image URI:', image_uri)

Created ECR repo: ana680-bgg-ratings
Image URI: 047557961235.dkr.ecr.us-east-2.amazonaws.com/ana680-bgg-ratings:latest


In [9]:
# Build and push container from notebook instance terminal environment
# Cleaner/quiet build + push logging
!chmod +x ../container/bgg_ratings/train ../container/bgg_ratings/serve
!aws ecr get-login-password --region {region} | docker login --username AWS --password-stdin {account_id}.dkr.ecr.{region}.amazonaws.com

# quiet build (prints final image id only)
!docker build -q -t {repo_name}:latest ../container/bgg_ratings

!docker tag {repo_name}:latest {image_uri}

# push output to log, then show only the tail
!docker push {image_uri} > /tmp/ecr_push.log 2>&1
!tail -n 30 /tmp/ecr_push.log


WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store

Login Succeeded
sha256:086ba3c4a66430bffebfc5669d17d945f6b10e7c2b720bbd11605996854a108f
The push refers to repository [047557961235.dkr.ecr.us-east-2.amazonaws.com/ana680-bgg-ratings]
2b8eb291faaf: Preparing
2b8eb291faaf: Preparing
5f70bf18a086: Preparing
278d8c443d5f: Preparing
6e97d78b14fb: Preparing
ff40d0a83ddf: Preparing
9ccec6a128bd: Preparing
3780204f6b74: Preparing
11eedb262098: Preparing
6400845f12ab: Preparing
a257f20c716c: Preparing
9ccec6a128bd: Waiting
3780204f6b74: Waiting
11eedb262098: Waiting
6400845f12ab: Waiting
a257f20c716c: Waiting
6e97d78b14fb: Layer already exists
278d8c443d5f: Layer already exists
ff40d0a83ddf: Layer already exists
2b8eb291faaf: Layer already exists
5f70bf18a086: Layer already exists
9ccec6a128bd: Layer already exists
378020

In [10]:
# Train on SageMaker using BYOC estimator
output_path = f's3://{bucket}/{prefix}/output'

estimator = Estimator(
    image_uri=image_uri,
    role=role,
    instance_count=1,
    instance_type='ml.m5.large',
    output_path=output_path,
    sagemaker_session=session,
)

estimator.fit({'train': train_s3_uri})
print('Training complete:', estimator.latest_training_job.name)
print('Model artifacts:', estimator.model_data)

INFO:sagemaker:Creating training-job with name: ana680-bgg-ratings-2026-03-01-09-07-00-070


2026-03-01 09:07:01 Starting - Starting the training job...
2026-03-01 09:07:15 Starting - Preparing the instances for training...
2026-03-01 09:07:36 Downloading - Downloading input data..Loading training data from: /opt/ml/input/data/train/bgg_master.csv
Saved model to: /opt/ml/model/model.pkl
Saved feature list to: /opt/ml/model/feature_columns.json
Saved metrics to: /opt/ml/output/data/metrics.json
{'rows': 21925, 'feature_count': 4, 'features': ['mech_roll_spin_and_move', 'GameWeight', 'Cat_War', 'YearPublished'], 'rmse': 0.6863869318093235, 'mae': 0.5160745166528892, 'r2': 0.48926026189688254, 'spearman': 0.028454258918296886}

2026-03-01 09:08:27 Training - Training image download completed. Training in progress.
2026-03-01 09:08:27 Uploading - Uploading generated training model
2026-03-01 09:08:41 Completed - Training job completed
Training seconds: 64
Billable seconds: 64
Training complete: ana680-bgg-ratings-2026-03-01-09-07-00-070
Model artifacts: s3://sagemaker-us-east-2-04

In [11]:
# Deploy endpoint for quick test
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type='ml.t2.medium',
)
predictor.serializer = CSVSerializer()
predictor.deserializer = JSONDeserializer()
print('Endpoint name:', predictor.endpoint_name)

INFO:sagemaker:Creating model with name: ana680-bgg-ratings-2026-03-01-09-09-45-580
INFO:sagemaker:Creating endpoint-config with name ana680-bgg-ratings-2026-03-01-09-09-45-580
INFO:sagemaker:Creating endpoint with name ana680-bgg-ratings-2026-03-01-09-09-45-580


---!Endpoint name: ana680-bgg-ratings-2026-03-01-09-09-45-580


In [12]:
# Prediction test using Terra Mystica-like values in feature order
# [mech_roll_spin_and_move, GameWeight, Cat_War, YearPublished]
sample = [[0, 3.9666, 0, 2012]]
pred = predictor.predict(sample)
print('Prediction output:', pred)

Prediction output: [7.316977024078369]


In [13]:
# Cleanup right after testing to avoid charges
predictor.delete_endpoint()
print('Endpoint deleted')

INFO:sagemaker:Deleting endpoint configuration with name: ana680-bgg-ratings-2026-03-01-09-09-45-580
INFO:sagemaker:Deleting endpoint with name: ana680-bgg-ratings-2026-03-01-09-09-45-580


Endpoint deleted


In [15]:
# Gut-check that the cleanup worked
sm = boto3.client('sagemaker', region_name=region)
for ep in sm.list_endpoints(SortBy='CreationTime', SortOrder='Descending', MaxResults=10).get('Endpoints', []):
    print(ep['EndpointName'], ep['EndpointStatus'])